<a href="https://colab.research.google.com/github/elangosaravanan21/sadtalkAI/blob/main/SadTalker_v0.2_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# 1. Reset and Setup
%cd /content
!rm -rf SadTalker
!git clone -b v2.0 https://github.com/camenduru/SadTalker
!pip install -q gradio safetensors kornia facexlib yacs gfpgan basicsr
%cd /content/SadTalker

# 2. THE TOTAL PATCH (Torchvision, NumPy, and All Gradio Errors)
# Fix Torchvision (BasicSR fix)
!sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/g' /usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py
!sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/g' /usr/local/lib/python3.12/dist-packages/basicsr/data/realesrgan_dataset.py

# Fix NumPy 2.0 AttributeError
!sed -i 's/category=np.VisibleDeprecationWarning/category=DeprecationWarning/g' /content/SadTalker/src/face3d/util/preprocess.py

# Fix Gradio Layout (Stripping all .style calls)
!sed -i 's/\.style([^)]*)//g' /content/SadTalker/app_sadtalker.py

# Fix Gradio Components (Removing deprecated 'source', 'tool', 'type', and 'format' arguments)
!sed -i 's/source="upload", //g' /content/SadTalker/app_sadtalker.py
!sed -i 's/tool="editor", //g' /content/SadTalker/app_sadtalker.py
!sed -i 's/type="filepath", //g' /content/SadTalker/app_sadtalker.py
!sed -i 's/format="mp4"//g' /content/SadTalker/app_sadtalker.py

print("All compatibility patches applied successfully!")

# 3. Download Models
!apt -y install -qq aria2
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/checkpoints/SadTalker_V0.0.2_256.safetensors -d /content/SadTalker/checkpoints -o SadTalker_V0.0.2_256.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/checkpoints/SadTalker_V0.0.2_512.safetensors -d /content/SadTalker/checkpoints -o SadTalker_V0.0.2_512.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/checkpoints/mapping_00109-model.pth.tar -d /content/SadTalker/checkpoints -o mapping_00109-model.pth.tar
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/checkpoints/mapping_00229-model.pth.tar -d /content/SadTalker/checkpoints -o mapping_00229-model.pth.tar
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/gfpgan/weights/GFPGANv1.4.pth -d /content/SadTalker/gfpgan/weights -o GFPGANv1.4.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/gfpgan/weights/alignment_WFLW_4HG.pth -d /content/SadTalker/gfpgan/weights -o alignment_WFLW_4HG.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/gfpgan/weights/detection_Resnet50_Final.pth -d /content/SadTalker/gfpgan/weights -o detection_Resnet50_Final.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/gfpgan/weights/parsing_parsenet.pth -d /content/SadTalker/gfpgan/weights -o parsing_parsenet.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/checkpoints/epoch_00190_iteration_000400000_checkpoint.pt -d /content/SadTalker/checkpoints -o epoch_00190_iteration_000400000_checkpoint.pt

# 4. Launch
!python app_sadtalker.py --share

/content
Cloning into 'SadTalker'...
remote: Enumerating objects: 1632, done.
remote: Total 1632 (delta 0), reused 0 (delta 0), pack-reused 1632 (from 1)
Receiving objects: 100% (1632/1632), 92.28 MiB | 13.88 MiB/s, done.
Resolving deltas: 100% (819/819), done.
/content/SadTalker
All compatibility patches applied successfully!
aria2 is already the newest version (1.36.0-1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
43afc8|OK  |   277MiB/s|/content/SadTalker/checkpoints/SadTalker_V0.0.2_256.safetensors

Status Legend:
(OK):download completed.

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
e03f4f|OK  |   249MiB/s|/content/SadTalker/checkpoints/SadTalker_V0.0.2_512.safetensors

Status Legend:
(OK):download completed.

Download Results:
gid   |stat|avg sp

In [ ]:
# Patch the inhomogeneous array error in face3d preprocess
!sed -i 's/trans_params = np.array(\[w0, h0, s, t\[0\], t\[1\]\])/trans_params = np.array([w0, h0, s, t[0], t[1]], dtype=object)/g' /content/SadTalker/src/face3d/util/preprocess.py

# Additional fix for a similar potential error in the same file
!sed -i 's/return np.array(\[pts, target_pts\])/return np.array([pts, target_pts], dtype=object)/g' /content/SadTalker/src/face3d/util/preprocess.py

print("NumPy 1D inhomogeneous array patch applied!")

# Re-launch
%cd /content/SadTalker
!python app_sadtalker.py --share

NumPy 1D inhomogeneous array patch applied!
/content/SadTalker
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://9df11914ca020c946e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
using safetensor as default
{'checkpoint': 'checkpoints/SadTalker_V0.0.2_256.safetensors', 'dir_of_BFM_fitting': 'src/config', 'audio2pose_yaml_path': 'src/config/auido2pose.yaml', 'audio2exp_yaml_path': 'src/config/auido2exp.yaml', 'pirender_yaml_path': 'src/config/facerender_pirender.yaml', 'pirender_checkpoint': 'checkpoints/epoch_00190_iteration_000400000_checkpoint.pt', 'use_safetensor': True, 'mappingnet_checkpoint': 'checkpoints/mapping_00229-model.pth.tar', 'facerender_yaml': 'src/config/facerender.yaml'}
/tmp/gradio/689671a3bc00c88e6a7703a5da195c0cec1e2ca848ae37501556271f098fe307/657340697_14712814576